# Session 3: empirical complexity and choosing a sorting method
**A. Real-world problem.** You are told "merge sort is O(n log n)". How do you know that is true for
*your* code and *your* data? You measure. In this session we count operations over a range of sizes,
fit straight lines on log-log axes, find the inputs that break a method, and then pick and justify a
method for a given data pattern. Operation counts carry the main claims because they do not depend on
the computer; wall-clock times are shown only as an illustration.

**Plan.** (1) the sorts with counters, (2) log-log slopes, (3) input patterns, (4) quicksort's worst
case and its cures, (5) a hybrid merge/insertion sort, (6) memory, (7) Python's built-in `sorted`,
(8) two failure cases, (9) the mini-project helper.

In [ ]:
import math
import pathlib
import random
import sys
import time
import tracemalloc

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the sorts, with counters
Each function returns `(sorted_list, comparisons, moves)`; `quick_sort` also returns the recursion depth.
*Moves* are writes of an item (swaps, shifts, or items copied into a merged list).

In [ ]:
def bubble_sort(values):
    a = list(values)
    comps = moves = 0
    for end in range(len(a) - 1, 0, -1):
        swapped = False
        for j in range(end):
            comps += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                moves += 1
                swapped = True
        if not swapped:
            break
    return a, comps, moves


def insertion_sort(values, key=None):
    k = key or (lambda x: x)
    a = list(values)
    comps = moves = 0
    for i in range(1, len(a)):
        item, j = a[i], i - 1
        while j >= 0:
            comps += 1
            if k(a[j]) > k(item):
                a[j + 1] = a[j]
                moves += 1
                j -= 1
            else:
                break
        a[j + 1] = item
    return a, comps, moves


def selection_sort(values, key=None):
    k = key or (lambda x: x)
    a = list(values)
    comps = moves = 0
    for i in range(len(a) - 1):
        m = i
        for j in range(i + 1, len(a)):
            comps += 1
            if k(a[j]) < k(a[m]):
                m = j
        if m != i:
            a[i], a[m] = a[m], a[i]
            moves += 1
    return a, comps, moves


def merge_sort(values, cutoff=1):
    """Top-down merge sort. Pieces of at most `cutoff` items are sorted by insertion sort (a hybrid)."""
    cnt = [0, 0]

    def sort(xs):
        if len(xs) <= max(cutoff, 1):
            if len(xs) > 1:
                out, c, m = insertion_sort(xs)
                cnt[0] += c
                cnt[1] += m
                return out
            return xs
        mid = len(xs) // 2
        left, right = sort(xs[:mid]), sort(xs[mid:])
        out, i, j = [], 0, 0
        while i < len(left) and j < len(right):
            cnt[0] += 1
            cnt[1] += 1
            if right[j] < left[i]:
                out.append(right[j])
                j += 1
            else:
                out.append(left[i])
                i += 1
        tail = left[i:] + right[j:]
        out.extend(tail)
        cnt[1] += len(tail)
        return out

    return sort(list(values)), cnt[0], cnt[1]

## Step 2: quicksort with three pivot rules
Quicksort picks a *pivot* item, moves smaller items before it, and sorts the two sides.
`first` always takes the first item. `random` takes a random item. `median3` takes the median of the first,
middle and last item (3 comparisons). The recursion depth is the number of nested pieces.

In [ ]:
def _choose(a, lo, hi, pivot, rng, cnt):
    if pivot == "first":
        return lo
    if pivot == "random":
        return rng.randint(lo, hi)
    mid = (lo + hi) // 2
    c1, c2, c3 = a[lo] < a[mid], a[mid] < a[hi], a[lo] < a[hi]
    cnt[0] += 3
    if c1 == c2:
        return mid
    return hi if c1 == c3 else lo


def _partition(a, lo, hi, pivot, rng, cnt):
    """Lomuto partition of a[lo..hi]; returns the final index of the pivot."""
    p = _choose(a, lo, hi, pivot, rng, cnt)
    if p != lo:
        a[lo], a[p] = a[p], a[lo]
        cnt[1] += 1
    v, i = a[lo], lo
    for j in range(lo + 1, hi + 1):
        cnt[0] += 1
        if a[j] < v:
            i += 1
            a[i], a[j] = a[j], a[i]
            cnt[1] += 1
    a[lo], a[i] = a[i], a[lo]
    cnt[1] += 1
    return i


def quick_sort(values, pivot="first", seed=0):
    """Return (sorted list, comparisons, moves, depth). Uses an explicit stack, so it never overflows."""
    a = list(values)
    rng = random.Random(seed)
    cnt = [0, 0]
    stack, depth = [(0, len(a) - 1, 1)], 0
    while stack:
        lo, hi, d = stack.pop()
        if lo >= hi:
            continue
        depth = max(depth, d)
        i = _partition(a, lo, hi, pivot, rng, cnt)
        stack.append((lo, i - 1, d + 1))
        stack.append((i + 1, hi, d + 1))
    return a, cnt[0], cnt[1], depth


def quick_recursive(a, lo, hi, pivot="first"):
    """The textbook recursive form, used only to demonstrate the failure case."""
    if lo >= hi:
        return
    i = _partition(a, lo, hi, pivot, random.Random(0), [0, 0])
    quick_recursive(a, lo, i - 1, pivot)
    quick_recursive(a, i + 1, hi, pivot)

## Step 3: input patterns
The same algorithm behaves differently on different data. Five patterns:
`random` (a shuffle of 0..n-1), `sorted`, `reversed`, `duplicates` (only 10 different values),
and `nearly` (sorted, then a few random swaps).

In [ ]:
def far_nearly_sorted(n, swaps, seed=1):
    """0..n-1 in order, then `swaps` random exchanges between any two places."""
    rng = random.Random(seed)
    a = list(range(n))
    for _ in range(swaps):
        i, j = rng.randrange(n), rng.randrange(n)
        a[i], a[j] = a[j], a[i]
    return a


def make_data(pattern, n, seed=0):
    rng = random.Random(seed)
    if pattern == "sorted":
        return list(range(n))
    if pattern == "reversed":
        return list(range(n - 1, -1, -1))
    if pattern == "duplicates":
        return [rng.randint(0, 9) for _ in range(n)]
    if pattern == "nearly":
        return far_nearly_sorted(n, max(1, n // 100), seed + 1)
    xs = list(range(n))
    rng.shuffle(xs)
    return xs


PATTERNS = ["random", "sorted", "reversed", "duplicates", "nearly"]


def run_method(name, data):
    """Return (comparisons, moves) for a method name on a list."""
    if name == "bubble":
        r = bubble_sort(data)
    elif name == "insertion":
        r = insertion_sort(data)
    elif name == "selection":
        r = selection_sort(data)
    elif name == "merge":
        r = merge_sort(data)
    elif name == "hybrid":
        r = merge_sort(data, 16)
    else:                                  # quick_first, quick_random, quick_median3
        r = quick_sort(data, name.split("_")[1])
    assert r[0] == sorted(data)
    return r[1], r[2]


METHODS = ["bubble", "selection", "insertion", "merge", "hybrid", "quick_first", "quick_random", "quick_median3"]


def cost(name, data):
    c, m = run_method(name, data)
    return c + m

## Step 4: slopes on log-log axes
If cost = a * n^p then log(cost) = log(a) + p log(n): a straight line of slope p.
`np.polyfit` finds the slope. For n log n the slope is not exactly 1 but 1 + 1/ln(n), a little above 1.

In [ ]:
def loglog_slope(ns, ys):
    return float(np.polyfit(np.log(ns), np.log(ys), 1)[0])


def count_curve(name, ns, pattern="random", seed=0):
    """Total comparisons for each n."""
    return [run_method(name, make_data(pattern, n, seed))[0] for n in ns]


QUAD_SIZES = [100, 200, 400, 800, 1600]
FAST_SIZES = [200, 400, 800, 1600, 3200, 6400, 12800]

## Step 5: Python's built-in `sorted` (Timsort) and timing
Timsort looks for runs that are already ascending or descending, so presorted input costs only n - 1
comparisons. We count comparisons with a small wrapper class and time it (timing is only illustrative).

In [ ]:
class Counted:
    count = 0

    def __init__(self, v):
        self.v = v

    def __lt__(self, other):
        Counted.count += 1
        return self.v < other.v


def timsort_comparisons(data):
    Counted.count = 0
    sorted(Counted(x) for x in data)
    return Counted.count


def best_time(fn, data, repeats=3):
    """Smallest wall-clock time (seconds) over a few repeats; the smallest is the least disturbed."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(list(data))
        best = min(best, time.perf_counter() - t0)
    return best

## Step 6: memory
Merge sort builds new lists, so it needs extra space of order n. Insertion, selection and bubble sort work in
place. `tracemalloc` reports the peak memory a call allocates; we subtract the cost of the copy every method makes.

In [ ]:
def extra_bytes(fn, data):
    """Peak extra bytes allocated by fn(data) beyond the plain copy of the list."""
    def peak(f):
        tracemalloc.start()
        f()
        p = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
        return p
    base = peak(lambda: list(data))
    return peak(lambda: fn(data)) - base

## Step 7: two failure cases
1. Naive recursive quicksort with a first-item pivot on sorted input recurses n deep and hits Python's limit.
2. A sort that is not stable scrambles records with equal keys (sorting by grade after sorting by name).

In [ ]:
def recursion_fails(n, pivot="first", margin=150):
    """True if recursive quicksort of the sorted list 0..n-1 exceeds a recursion limit `margin` frames above now."""
    frames, f = 0, sys._getframe()
    while f is not None:
        frames += 1
        f = f.f_back
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(frames + margin)
    try:
        quick_recursive(list(range(n)), 0, n - 1, pivot)
        return False
    except RecursionError:
        return True
    finally:
        sys.setrecursionlimit(old)


def name_then_grade(sorter):
    """Sort by name, then by grade with `sorter` (key-aware). Return the final list of (grade, name)."""
    people = [(3, "Asha"), (1, "Bala"), (3, "Chitra"), (1, "Devi"), (3, "Esha"), (2, "Farid"), (1, "Gita")]
    by_name = sorted(people, key=lambda r: r[1])
    return sorter(by_name, key=lambda r: r[0])[0]

## Step 8: helpers for the mini-project
Which method is cheapest for a given pattern? `crossover_swaps` finds how many far swaps in a sorted list
make merge sort cheaper than insertion sort.

In [ ]:
def crossover_swaps(n=2000, limit=200, seed=1, cutoff=1):
    """Smallest number k of random far swaps for which merge sort (with the given cutoff) costs less
    than insertion sort, counting comparisons + moves."""
    for k in range(limit + 1):
        data = far_nearly_sorted(n, k, seed)
        if sum(merge_sort(data, cutoff)[1:]) < cost("insertion", data):
            return k
    return None


def best_cutoff(n=4096, cutoffs=(1, 2, 4, 8, 16, 32, 64, 128, 256), seed=0):
    """(cutoff with the lowest total cost, dict cutoff -> cost) for hybrid merge sort on random data."""
    data = make_data("random", n, seed)
    table = {c: sum(merge_sort(data, c)[1:]) for c in cutoffs}
    return min(table, key=table.get), table


def small_n_crossover(seeds=10, nmax=64):
    """Smallest n (2..nmax) at which pure merge sort costs less than insertion sort on average random data."""
    for n in range(2, nmax + 1):
        mi = sum(sum(insertion_sort(make_data("random", n, s))[1:]) for s in range(seeds))
        mm = sum(sum(merge_sort(make_data("random", n, s))[1:]) for s in range(seeds))
        if mm < mi:
            return n
    return None

## Step 9: see it
Figure 1: comparisons against n on log-log axes with fitted slopes. Figure 2: quicksort on sorted input
with three pivot rules. Figure 3: total cost of hybrid merge sort against the cutoff.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for name, sizes in (("bubble", QUAD_SIZES), ("selection", QUAD_SIZES), ("insertion", QUAD_SIZES),
                        ("merge", FAST_SIZES), ("quick_random", FAST_SIZES)):
        ys = count_curve(name, sizes)
        s = loglog_slope(sizes, ys)
        ax.loglog(sizes, ys, "o-", ms=4, label="%s (slope %.2f)" % (name, s))
        print("%-13s slope %.3f" % (name, s))
    ax.set_xlabel("n (items, log scale)")
    ax.set_ylabel("comparisons (log scale)")
    ax.set_title("Straight lines on log-log axes: the slope is the exponent")
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "loglog_slopes.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4.5))
    for piv in ("first", "random", "median3"):
        ys = [quick_sort(make_data("sorted", n), piv)[1] for n in QUAD_SIZES]
        ax2.loglog(QUAD_SIZES, ys, "o-", ms=4, label="pivot %s (slope %.2f)" % (piv, loglog_slope(QUAD_SIZES, ys)))
        print("quicksort on sorted input, pivot", piv, ys)
    ax2.set_xlabel("n (items, log scale)")
    ax2.set_ylabel("comparisons (log scale)")
    ax2.set_title("Quicksort on sorted input: the pivot rule decides")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "quicksort_pivots.png", dpi=120)

    c_best, table = best_cutoff()
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ax3.semilogx(list(table), list(table.values()), "o-")
    ax3.set_xlabel("cutoff: pieces up to this size use insertion sort")
    ax3.set_ylabel("comparisons + moves (n = 4096)")
    ax3.set_title("Hybrid merge sort: best cutoff %d" % c_best)
    ax3.grid(alpha=0.3, which="both")
    fig3.tight_layout()
    fig3.savefig(OUT / "hybrid_cutoff.png", dpi=120)
    print("hybrid table", table)
    print("small-n crossover", small_n_crossover(), "| far-swap crossover", crossover_swaps())
    print("n = 1000 total cost by pattern")
    for pat in PATTERNS:
        print(" ", pat, {m: cost(m, make_data(pat, 1000)) for m in METHODS})
    data = list(range(20000))
    print("timsort comparisons, sorted 20000:", timsort_comparisons(data))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `count_curve` and `loglog_slope` to measure the slope of selection sort on the pattern 'sorted' for n from 100 to 1600 and explain the result.
2. **Stretch.** Add a `quick_sort` pivot rule 'middle' (the middle item) and test it on the five patterns; does it survive the sorted input? Does it survive the reversed input?
3. **Explain.** Explain why the best cutoff in `best_cutoff` is small and not 64, using the cost of insertion sort on a piece of size m.

## Assessment
- **A. Conceptual.** Why is a sorted list the worst case for quicksort with the first item as pivot, and why does that also create a risk of crashing, not just of being slow?
- **B. Numerical.** The comparisons for insertion sort on a reversed list are 4950 for n = 100 and 19,900 for n = 200. Compute the log-log slope between these two points and say what it tells you.
- **C. Graph and simulation interpretation.** On the log-log figure, which lines are parallel, what are their approximate slopes, and what does the gap between the two groups of lines mean?
- **D. Design and optimisation challenge.** Design: 1000 values contain only 10 distinct numbers (many duplicates). Choose among merge sort, quicksort with a first-item pivot and insertion sort, and justify your choice with costs. What does this tell you about the pivot rule?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math
import random
import struct


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # validation against sorted(): every method, every pattern, plus tiny and degenerate inputs
    ok = True
    for pat in L.PATTERNS:
        for name in L.METHODS:
            data = L.make_data(pat, 300, 4)
            snap = list(data)
            L.run_method(name, data)          # run_method asserts the output equals sorted(data)
            ok &= data == snap
    for name in L.METHODS:
        for data in ([], [5], [2, 1], [7] * 50):
            L.run_method(name, data)
    checks.append(("all 8 methods match sorted() on 5 patterns, on empty, single, pair and all-equal inputs", ok))

    # slopes on log-log axes (counts, so independent of the computer)
    s = {}
    for name in ("bubble", "selection", "insertion"):
        s[name] = L.loglog_slope(L.QUAD_SIZES, L.count_curve(name, L.QUAD_SIZES))
    for name in ("merge", "hybrid", "quick_random", "quick_median3"):
        s[name] = L.loglog_slope(L.FAST_SIZES, L.count_curve(name, L.FAST_SIZES))
    checks.append(("slopes of bubble, selection and insertion comparisons are between 1.9 and 2.1 (about 2)",
                   all(1.9 < s[k] < 2.1 for k in ("bubble", "selection", "insertion"))))
    checks.append(("slopes of merge, hybrid and quick (random or median-of-3 pivot) are between 1.05 and 1.25 (1 plus a little)",
                   all(1.05 < s[k] < 1.25 for k in ("merge", "hybrid", "quick_random", "quick_median3"))))
    pure = 1 + 1 / math.log(1600)
    checks.append(("hand check: n log n has local slope 1 + 1/ln n, about 1.14 near n = 1600, close to the fitted merge slope",
                   abs(pure - 1.1355) < 0.001 and abs(s["merge"] - pure) < 0.08))

    # quicksort worst case and its cures: worked example (Section E)
    first = L.quick_sort(L.make_data("sorted", 200), "first")
    med = L.quick_sort(L.make_data("sorted", 200), "median3")
    rnd = L.quick_sort(L.make_data("sorted", 200), "random")
    checks.append(("worked example: sorted input of 200, first pivot: 19,900 = 200*199/2 comparisons, 199 moves, depth 199",
                   first[1:] == (19900, 199, 199) and first[1] == 200 * 199 // 2))
    checks.append(("worked example: sorted input of 200, median-of-3 pivot: 1,534 comparisons, depth 7",
                   med[1] == 1534 and med[3] == 7 and med[2] == 734))
    checks.append(("worked example: sorted input of 200, random pivot (seed 0): 1,625 comparisons, depth 16",
                   rnd[1] == 1625 and rnd[3] == 16))
    checks.append(("on sorted input the first-pivot depth is n - 1 for every size tested",
                   all(L.quick_sort(L.make_data("sorted", n), "first")[3] == n - 1 for n in L.QUAD_SIZES)))
    s_first = L.loglog_slope(L.QUAD_SIZES, [L.quick_sort(L.make_data("sorted", n), "first")[1] for n in L.QUAD_SIZES])
    s_med = L.loglog_slope(L.QUAD_SIZES, [L.quick_sort(L.make_data("sorted", n), "median3")[1] for n in L.QUAD_SIZES])
    checks.append(("quicksort on sorted input: slope about 2 with a first pivot, below 1.3 with median-of-3",
                   1.9 < s_first < 2.1 and s_med < 1.3))
    allsame = L.quick_sort([7] * 200, "first")
    checks.append(("all-equal input is also a worst case for this quicksort: 19,900 comparisons for 200 items", allsame[1] == 19900))
    rev_med = L.cost("quick_median3", L.make_data("reversed", 1000))
    rev_rnd = L.cost("quick_random", L.make_data("reversed", 1000))
    checks.append(("median-of-3 is not a guarantee: on reversed input of 1000 it costs 190,506, over 5 times the random pivot",
                   rev_med == 190_506 and rev_med > 5 * rev_rnd))

    # input patterns at n = 1000
    tab = {pat: {m: L.cost(m, L.make_data(pat, 1000)) for m in L.METHODS} for pat in L.PATTERNS}
    checks.append(("random input n = 1000: merge costs 18,679 and bubble sort over 20 times more",
                   tab["random"]["merge"] == 18_679 and tab["random"]["bubble"] > 20 * tab["random"]["merge"]))
    checks.append(("sorted input n = 1000: bubble and insertion cost only 999; selection still costs 499,500",
                   tab["sorted"]["bubble"] == 999 and tab["sorted"]["insertion"] == 999 and tab["sorted"]["selection"] == 499_500))
    checks.append(("many duplicates (10 values): quick sort with a first pivot costs 57,190, over 3 times merge sort (18,516)",
                   tab["duplicates"]["quick_first"] == 57_190 and tab["duplicates"]["merge"] == 18_516
                   and tab["duplicates"]["quick_first"] > 3 * tab["duplicates"]["merge"]))
    checks.append(("nearly sorted (10 far swaps in 1000): insertion costs 16,159 and quick (first pivot) is about 9 times worse",
                   tab["nearly"]["insertion"] == 16_159 and tab["nearly"]["quick_first"] > 8 * tab["nearly"]["insertion"]))

    checks.append(("worked example: first-pivot comparisons are about 13 times the median-of-3 comparisons (19,900 / 1,534)",
                   round(19900 / 1534) == 13))
    checks.append(("duplicates: insertion sort costs over 100,000 at n = 1000", tab["duplicates"]["insertion"] > 100_000))

    # hybrid merge sort and cross-over sizes
    cut, table = L.best_cutoff()
    checks.append(("hybrid: the cheapest cutoff for n = 4096 is 4 (88,172) against 93,154 for pure merge sort",
                   cut == 4 and table[4] == 88_172 and table[1] == 93_154))
    checks.append(("hybrid: a cutoff of 64 is worse than pure merge sort (182,555 against 93,154)", table[64] == 182_555 > table[1]))
    checks.append(("cross-over size: for lists of 11 items or more merge sort needs fewer steps than insertion sort", L.small_n_crossover() == 11))

    # memory
    word = struct.calcsize("P")
    n = 20000
    data = L.make_data("random", n)
    m_merge = L.extra_bytes(L.merge_sort, data)
    m_ins = L.extra_bytes(L.insertion_sort, L.make_data("random", 2000))
    checks.append(("memory: merge sort allocates more than n extra slots; insertion sort allocates under 0.1 n",
                   m_merge > word * n and m_ins < 0.1 * word * 2000))
    sel = L.selection_sort(L.make_data("random", 1024))
    mer = L.merge_sort(L.make_data("random", 1024))
    ins = L.insertion_sort(L.make_data("random", 1024))
    checks.append(("moves: selection sort moves at most n - 1 = 1023 items; merge sort 10,240; insertion sort far more",
                   sel[2] <= 1023 and mer[2] == 10_240 and ins[2] > 200_000))

    # Python's built-in sorted (Timsort)
    srt = list(range(20000))
    rand = L.make_data("random", 20000)
    checks.append(("timsort: sorted and reversed input of 20,000 need exactly 19,999 comparisons",
                   L.timsort_comparisons(srt) == 19_999 and L.timsort_comparisons(srt[::-1]) == 19_999))
    c = L.timsort_comparisons(rand)
    checks.append(("timsort: random input of 20,000 needs between 0.5 and 1.0 times n log2 n comparisons",
                   0.5 * 20000 * math.log2(20000) < c < 20000 * math.log2(20000)))
    checks.append(("timsort: an empty list needs 0 comparisons", L.timsort_comparisons([]) == 0))
    big = list(range(200000))
    shuf = big[:]
    random.Random(1).shuffle(shuf)
    t_sorted, t_random = L.best_time(sorted, big), L.best_time(sorted, shuf)
    checks.append(("timing (illustrative): sorted() on shuffled data takes over 3 times longer than on presorted data", t_random > 3 * t_sorted))

    # timing: slopes in time are only illustrative
    tb = [L.best_time(L.bubble_sort, L.make_data("random", k)) for k in (400, 1600)]
    tm = [L.best_time(L.merge_sort, L.make_data("random", k)) for k in (400, 1600)]
    checks.append(("timing (illustrative): 4 times more items makes bubble sort over 6 times slower, merge sort under 10 times",
                   tb[1] / tb[0] > 6 and tm[1] / tm[0] < 10))
    checks.append(("timing (illustrative): bubble sort of 1600 items is slower than merge sort", tb[1] > tm[1]))

    # failure cases
    checks.append(("failure: recursive quicksort (first pivot) on sorted input of 1000 exceeds the recursion limit", L.recursion_fails(1000, "first")))
    checks.append(("failure: median-of-3 pivot survives the same input, and 100 sorted items are fine even with a first pivot",
                   not L.recursion_fails(1000, "median3") and not L.recursion_fails(100, "first")))
    good = L.name_then_grade(L.insertion_sort)
    bad = L.name_then_grade(L.selection_sort)
    checks.append(("failure: insertion sort is stable (grade 3 stays Asha, Chitra, Esha); selection sort gives Esha, Asha, Chitra",
                   [r[1] for r in good if r[0] == 3] == ["Asha", "Chitra", "Esha"] and [r[1] for r in bad if r[0] == 3] == ["Esha", "Asha", "Chitra"]))

    # numbers quoted in the roadmap (what the shipped code does)
    d10 = L.far_nearly_sorted(2000, 10)
    dup = L.make_data("duplicates", 1000)
    worst = max(L.run_method("quick_first", L.make_data(p, 1000))[0] for p in L.PATTERNS)
    checks.append(("roadmap baselines: best shipped comparisons 13,565 on 2000 items with 10 far swaps; first-pivot quick sort 54,365 on duplicates and 499,500 worst",
                   L.merge_sort(d10, 4)[1] == 13_565 and L.insertion_sort(d10)[1] == 17_189 and L.merge_sort(d10)[1] == 14_019
                   and L.run_method("quick_first", dup)[0] == 54_365 and worst == 499_500))
    checks.append(("hand check: doubling n from 100 to 200 multiplies n(n-1)/2 from 4950 to 19,900, a slope of log2(4.02) = 2.007",
                   abs(math.log2(19900 / 4950) - 2.007) < 0.001))

    # mini-project helper
    checks.append(("mini-project: with n = 2000, merge sort beats insertion sort from 11 far swaps; a hybrid with cutoff 4 from 10",
                   L.crossover_swaps() == 11 and L.crossover_swaps(cutoff=4) == 10))
    few = L.far_nearly_sorted(2000, 2)
    c_ins = L.cost("insertion", few)
    c_hyb = sum(L.merge_sort(few, 4)[1:])
    checks.append(("mini-project: with 2 far swaps insertion sort costs 5,923, more than 4 times cheaper than the hybrid (29,384)",
                   c_ins == 5923 and c_hyb == 29_384 and c_hyb > 4 * c_ins))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
